# Optimize a Four-Bar Linkage with PSO

This notebook demonstrates the full optimization loop: define a four-bar linkage,
write a fitness function, and use Particle Swarm Optimization to find the best
link lengths for a given objective.

**Objective:** Maximize the horizontal range of the coupler point trajectory
(useful for conveyor, pick-and-place, or walking mechanisms).

**What you'll learn:**
- Building a parameterized linkage
- Writing a fitness function with `@kinematic_maximization`
- Running PSO with `particle_swarm_optimization()`
- Comparing initial vs optimized coupler curves

In [ ]:
import warnings

import matplotlib.pyplot as plt

from pylinkage.optimization import (
    chain_optimizers,
    dual_annealing_optimization,
    generate_bounds,
    minimize_linkage,
    particle_swarm_optimization,
)
from pylinkage.optimization.utils import kinematic_maximization
from pylinkage.synthesis import fourbar_from_lengths
from pylinkage.visualizer import plot_static_linkage

warnings.filterwarnings('ignore', category=DeprecationWarning)

## 1. Build the Initial Four-Bar

A simple crank-rocker with ground length 4. We'll optimize the crank (a),
coupler (b), and rocker (c) lengths.

In [ ]:
GROUND_LENGTH = 4.0

def build_fourbar(a, b, c):
    """Build a four-bar linkage with given link lengths."""
    return fourbar_from_lengths(
        crank_length=a,
        coupler_length=b,
        rocker_length=c,
        ground_length=GROUND_LENGTH,
        iterations=100,
    )

# Initial link lengths
initial = (1.0, 3.0, 3.0)
linkage = build_fourbar(*initial)
print(f'Joints: {[j.name for j in linkage.components]}')
print(f'Constraints: {linkage.get_num_constraints()}')

In [ ]:
# Simulate and plot the initial coupler curve with mechanism
loci = list(linkage.step(iterations=100))
c_path = [(pos[-1][0], pos[-1][1]) for pos in loci if pos[-1][0] is not None]
cx, cy = zip(*c_path, strict=False)

fig, ax = plt.subplots(figsize=(8, 6))

# Draw mechanism
plot_static_linkage(linkage, ax, loci, show_legend=True, show_labels=True,
                    show_loci=False,
                    title=f'Initial four-bar (a={initial[0]}, b={initial[1]}, c={initial[2]})')

# Coupler curve
ax.plot(cx, cy, 'r-', linewidth=2, label='Coupler curve (C)')
ax.legend()
plt.show()

x_range = max(cx) - min(cx)
print(f'Horizontal range: {x_range:.3f}')

## 2. Define the Fitness Function

The `@kinematic_maximization` decorator handles:
- Setting constraints on the linkage
- Running the simulation
- Catching `UnbuildableError` (returns `-inf`)

The decorated function receives the simulation loci and returns the score to maximize.

In [ ]:
@kinematic_maximization
def horizontal_range(loci, **kwargs):
    """Maximize horizontal range of the coupler point trajectory."""
    # Last joint is C (the coupler-rocker joint)
    c_path = [(pos[-1][0], pos[-1][1]) for pos in loci
              if pos[-1][0] is not None]
    if len(c_path) < 10:
        return 0.0
    xs = [p[0] for p in c_path]
    return max(xs) - min(xs)

# Quick test
score = horizontal_range(linkage, list(initial), None)
print(f"Initial fitness: {score:.3f}")

## 3. Run Particle Swarm Optimization

PSO explores the parameter space with a swarm of candidate solutions.
`generate_bounds()` creates search bounds around the initial values.

In [ ]:
center = list(initial)
lower, upper = generate_bounds(center, min_ratio=3, max_factor=3)
print('Search bounds:')
print(f'  Lower: {lower}')
print(f'  Upper: {upper}')

results = particle_swarm_optimization(
    horizontal_range,
    linkage,
    center=center,
    dimensions=len(center),
    n_particles=40,
    iters=80,
    bounds=(tuple(lower), tuple(upper)),
    order_relation=max,
    verbose=True,
)

best = results[0]
print(f'\nBest score: {best.score:.4f}')
print(
    f'Best dimensions: a={best.dimensions[0]:.3f}, '
    f'b={best.dimensions[1]:.3f}, c={best.dimensions[2]:.3f}'
)

## 4. Compare Before and After

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

def plot_coupler(ax, a, b, c, title):
    lk = build_fourbar(a, b, c)
    loci = list(lk.step(iterations=100))
    path = [(pos[-1][0], pos[-1][1]) for pos in loci if pos[-1][0] is not None]

    # Draw mechanism
    plot_static_linkage(lk, ax, loci, show_legend=True, show_labels=False,
                        show_loci=False)

    if path:
        px, py = zip(*path, strict=False)
        ax.plot(px, py, 'r-', linewidth=2, label='Coupler curve')
        x_range = max(px) - min(px)
        ax.set_title(f'{title}\nX range: {x_range:.3f}')

    ax.legend(fontsize=8)

plot_coupler(ax1, *initial, 'Initial (a=1, b=3, c=3)')
d = best.dimensions
plot_coupler(ax2, d[0], d[1], d[2],
             f'Optimized (a={d[0]:.2f}, b={d[1]:.2f}, c={d[2]:.2f})')

plt.tight_layout()
plt.show()

## 5. Overlay on Same Axes

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Initial
lk1 = build_fourbar(*initial)
loci1 = list(lk1.step(iterations=100))
p1 = [(pos[-1][0], pos[-1][1]) for pos in loci1 if pos[-1][0] is not None]
if p1:
    ax.plot(*zip(*p1, strict=False), 'b--', linewidth=1.5, alpha=0.6, label='Initial curve')
# Draw initial mechanism (faded)
plot_static_linkage(lk1, ax, loci1, show_legend=False, show_labels=False,
                    show_loci=False)
# Fade the initial mechanism lines
for line in ax.get_lines()[-10:]:
    if line.get_linewidth() == 3:
        line.set_alpha(0.5)
        line.set_linestyle('--')

# Optimized
lk2 = build_fourbar(d[0], d[1], d[2])
loci2 = list(lk2.step(iterations=100))
p2 = [(pos[-1][0], pos[-1][1]) for pos in loci2 if pos[-1][0] is not None]
if p2:
    ax.plot(*zip(*p2, strict=False), 'r-', linewidth=2, label='Optimized curve')
# Draw optimized mechanism
plot_static_linkage(lk2, ax, loci2, show_legend=True, show_labels=False,
                    show_loci=False, title='Coupler curves: initial vs optimized')

ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Interactive Exploration

Use the slider to drive the optimized linkage and explore its motion interactively.

In [ ]:
from pylinkage.visualizer import interactive_linkage_plotly

# Build the optimized linkage for interactive exploration
opt_linkage = build_fourbar(d[0], d[1], d[2])
interactive_linkage_plotly(opt_linkage, title="Optimized four-bar — drag the slider")


## 7. Dual Annealing

Dual annealing is a global optimizer based on generalized simulated annealing.
Unlike PSO (which maintains a swarm of particles), it follows a **single trajectory**
with controlled random jumps, making it effective for problems with many local
minima and relatively expensive evaluations.

In [ ]:
# Rebuild a fresh linkage so we start from the same initial state
da_linkage = build_fourbar(*initial)

da_results = dual_annealing_optimization(
    horizontal_range,
    da_linkage,
    bounds=(tuple(lower), tuple(upper)),
    order_relation=max,
    maxiter=200,
    seed=42,
    verbose=False,
)

da_best = da_results[0]
print(f'Dual annealing score: {da_best.score:.4f}')
print(
    f'Dimensions: a={da_best.dimensions[0]:.3f}, '
    f'b={da_best.dimensions[1]:.3f}, c={da_best.dimensions[2]:.3f}'
)
print(f'\nPSO score for comparison: {best.score:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# PSO result
lk_pso = build_fourbar(d[0], d[1], d[2])
loci_pso = list(lk_pso.step(iterations=100))
p_pso = [(pos[-1][0], pos[-1][1]) for pos in loci_pso if pos[-1][0] is not None]
if p_pso:
    ax.plot(*zip(*p_pso, strict=False), 'r-', linewidth=2, label=f'PSO (score={best.score:.3f})')

# Dual annealing result
da_d = da_best.dimensions
lk_da = build_fourbar(da_d[0], da_d[1], da_d[2])
loci_da = list(lk_da.step(iterations=100))
p_da = [(pos[-1][0], pos[-1][1]) for pos in loci_da if pos[-1][0] is not None]
if p_da:
    ax.plot(*zip(*p_da, strict=False), 'g--', linewidth=2,
            label=f'Dual annealing (score={da_best.score:.3f})')

# Draw DA mechanism at initial position
plot_static_linkage(lk_da, ax, loci_da, show_legend=True, show_labels=False,
                    show_loci=False, title='Coupler curves: PSO vs Dual Annealing')

ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 8. Optimizer Chaining

A common optimization strategy is to **chain** a global search with a local
refinement step. `chain_optimizers()` runs a sequence of optimizers, passing the
best solution from each stage as the starting point for the next.

Here we use dual annealing for broad exploration, then Nelder-Mead simplex to
polish the result into a tighter local optimum.

In [ ]:
# Rebuild a fresh linkage for the chained run
chain_linkage = build_fourbar(*initial)

chain_results = chain_optimizers(
    eval_func=horizontal_range,
    linkage=chain_linkage,
    stages=[
        (dual_annealing_optimization, {
            "bounds": (tuple(lower), tuple(upper)),
            "maxiter": 100,
            "seed": 42,
            "verbose": False,
        }),
        (minimize_linkage, {
            "method": "L-BFGS-B",
            "bounds": (tuple(lower), tuple(upper)),
            "maxiter": 200,
            "verbose": False,
        }),
    ],
    order_relation=max,
    verbose=False,
)

chain_best = chain_results[0]
print(f'Chained score:         {chain_best.score:.4f}')
print(f'Standalone DA score:   {da_best.score:.4f}')
print(f'PSO score:             {best.score:.4f}')
print(
    f'\nChained dimensions: a={chain_best.dimensions[0]:.3f}, '
    f'b={chain_best.dimensions[1]:.3f}, c={chain_best.dimensions[2]:.3f}'
)

## Summary

1. Built a parameterized four-bar linkage with `fourbar_from_lengths()`
2. Defined a fitness function with `@kinematic_maximization`
3. Ran PSO with `particle_swarm_optimization()` to maximize coupler X range
4. Compared coupler curves before and after optimization
5. Ran `dual_annealing_optimization()` as an alternative global optimizer
6. Used `chain_optimizers()` to combine dual annealing with Nelder-Mead local refinement

**Other objectives you can optimize for:**
- Path straightness (minimize y deviation)
- Proximity to target points
- Transmission angle quality
- Minimum link length ratios (for manufacturability)

**Other optimizers available:**
- `trials_and_errors_optimization()` -- grid search
- `differential_evolution_optimization()` -- scipy DE
- `multi_objective_optimization()` -- Pareto front (requires `pymoo`)